# Lab 08: KNN Algorithm & Support Vector Machines
**Name:** Abdul Hadi Saqib  
**CMS:** 467626

## Task 1: Data Loading and Preprocessing

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from collections import Counter

In [4]:
df = pd.read_csv('data/lab8/WineQT.csv')
print("Dataset Shape:", df.shape)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data/lab8/WineQT.csv'

In [ ]:
print("Missing Values:")
print(df.isnull().sum())
print("\nTarget Classes:", df['quality'].unique())

In [ ]:
X = df.drop(['quality', 'Id'], axis=1)
y = df['quality']

featureNames = X.columns.tolist()
print("Features:", featureNames)

In [ ]:
xTrain, xTest, yTrain, yTest = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
xTrainScaled = scaler.fit_transform(xTrain)
xTestScaled = scaler.transform(xTest)

print(f"Training set: {xTrainScaled.shape[0]} samples")
print(f"Testing set: {xTestScaled.shape[0]} samples")

## Task 2: Implementation of KNN from Scratch

In [ ]:
def euclideanDistance(point1, point2):
    return np.sqrt(np.sum((point1 - point2) ** 2))

def knnPredict(xTrainData, yTrainData, testInstance, k=3):
    distances = []
    for i in range(len(xTrainData)):
        dist = euclideanDistance(testInstance, xTrainData[i])
        distances.append((dist, yTrainData.iloc[i]))
    distances.sort(key=lambda x: x[0])
    neighbors = [d[1] for d in distances[:k]]
    return Counter(neighbors).most_common(1)[0][0]

def knnPredictAll(xTrainData, yTrainData, xTestData, k=3):
    predictions = []
    for testInstance in xTestData:
        pred = knnPredict(xTrainData, yTrainData, testInstance, k)
        predictions.append(pred)
    return np.array(predictions)

In [ ]:
twoFeatures = ['alcohol', 'volatile acidity']
xTrain2F = xTrain[twoFeatures].values
xTest2F = xTest[twoFeatures].values

scaler2F = StandardScaler()
xTrain2FScaled = scaler2F.fit_transform(xTrain2F)
xTest2FScaled = scaler2F.transform(xTest2F)

yPredScratch2F = knnPredictAll(xTrain2FScaled, yTrain.reset_index(drop=True), xTest2FScaled, k=5)
accScratch2F = accuracy_score(yTest, yPredScratch2F)
print(f"KNN from Scratch (2 Features) Accuracy: {accScratch2F:.4f}")

In [ ]:
plt.figure(figsize=(10, 6))
classes = np.unique(yTrain)
colors = plt.cm.viridis(np.linspace(0, 1, len(classes)))

for cls, color in zip(classes, colors):
    mask = yTrain.reset_index(drop=True) == cls
    plt.scatter(xTrain2FScaled[mask, 0], xTrain2FScaled[mask, 1], 
                c=[color], label=f'Class {cls}', alpha=0.6, marker='o')

testIdx = 0
plt.scatter(xTest2FScaled[testIdx, 0], xTest2FScaled[testIdx, 1], 
            c='red', marker='*', s=300, edgecolors='black', 
            label=f'Test Point (Pred: {yPredScratch2F[testIdx]})')

plt.xlabel(twoFeatures[0])
plt.ylabel(twoFeatures[1])
plt.title('KNN from Scratch - 2 Features Visualization')
plt.legend()
plt.show()

In [ ]:
threeFeatures = ['alcohol', 'volatile acidity', 'citric acid']
xTrain3F = xTrain[threeFeatures].values
xTest3F = xTest[threeFeatures].values

scaler3F = StandardScaler()
xTrain3FScaled = scaler3F.fit_transform(xTrain3F)
xTest3FScaled = scaler3F.transform(xTest3F)

yPredScratch3F = knnPredictAll(xTrain3FScaled, yTrain.reset_index(drop=True), xTest3FScaled, k=5)
accScratch3F = accuracy_score(yTest, yPredScratch3F)
print(f"KNN from Scratch (3 Features) Accuracy: {accScratch3F:.4f}")

In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

for cls, color in zip(classes, colors):
    mask = yTrain.reset_index(drop=True) == cls
    ax.scatter(xTrain3FScaled[mask, 0], xTrain3FScaled[mask, 1], xTrain3FScaled[mask, 2],
               c=[color], label=f'Class {cls}', alpha=0.6)

ax.scatter(xTest3FScaled[testIdx, 0], xTest3FScaled[testIdx, 1], xTest3FScaled[testIdx, 2],
           c='red', marker='*', s=300, label=f'Test Point (Pred: {yPredScratch3F[testIdx]})')

ax.set_xlabel(threeFeatures[0])
ax.set_ylabel(threeFeatures[1])
ax.set_zlabel(threeFeatures[2])
ax.set_title('KNN from Scratch - 3 Features Visualization')
ax.legend()
plt.show()

## Task 3: KNN Using Scikit-Learn

In [ ]:
kValues = [3, 5, 7]

print("KNN with 2 Features:")
for k in kValues:
    knnModel = KNeighborsClassifier(n_neighbors=k)
    knnModel.fit(xTrain2FScaled, yTrain)
    yPred = knnModel.predict(xTest2FScaled)
    acc = accuracy_score(yTest, yPred)
    print(f"  K={k}: Accuracy = {acc:.4f}")

In [ ]:
print("KNN with 3 Features:")
for k in kValues:
    knnModel = KNeighborsClassifier(n_neighbors=k)
    knnModel.fit(xTrain3FScaled, yTrain)
    yPred = knnModel.predict(xTest3FScaled)
    acc = accuracy_score(yTest, yPred)
    print(f"  K={k}: Accuracy = {acc:.4f}")

In [ ]:
fourFeatures = ['alcohol', 'volatile acidity', 'citric acid', 'sulphates']
xTrain4F = xTrain[fourFeatures].values
xTest4F = xTest[fourFeatures].values

scaler4F = StandardScaler()
xTrain4FScaled = scaler4F.fit_transform(xTrain4F)
xTest4FScaled = scaler4F.transform(xTest4F)

print("KNN with 4 Features:")
for k in kValues:
    knnModel = KNeighborsClassifier(n_neighbors=k)
    knnModel.fit(xTrain4FScaled, yTrain)
    yPred = knnModel.predict(xTest4FScaled)
    acc = accuracy_score(yTest, yPred)
    print(f"  K={k}: Accuracy = {acc:.4f}")

In [ ]:
knnSklearn = KNeighborsClassifier(n_neighbors=5)
knnSklearn.fit(xTrain2FScaled, yTrain)
yPredSklearn2F = knnSklearn.predict(xTest2FScaled)

plt.figure(figsize=(10, 6))
for cls, color in zip(classes, colors):
    mask = yTest.reset_index(drop=True) == cls
    plt.scatter(xTest2FScaled[mask, 0], xTest2FScaled[mask, 1], 
                c=[color], label=f'Class {cls}', alpha=0.7)

plt.xlabel(twoFeatures[0])
plt.ylabel(twoFeatures[1])
plt.title('KNN Sklearn - Test Set Predictions (2 Features)')
plt.legend()
plt.show()

print(f"\nScratch vs Sklearn Comparison (2 Features, K=5):")
print(f"  Scratch Accuracy: {accScratch2F:.4f}")
print(f"  Sklearn Accuracy: {accuracy_score(yTest, yPredSklearn2F):.4f}")

## Task 4: Support Vector Machine (SVM) Classification

In [ ]:
kernels = ['linear', 'rbf']
cValues = [0.1, 1, 10]
gammaValues = ['scale', 'auto']

print("SVM with Different Configurations (2 Features):")
for kernel in kernels:
    for c in cValues:
        if kernel == 'rbf':
            for gamma in gammaValues:
                svmModel = SVC(kernel=kernel, C=c, gamma=gamma)
                svmModel.fit(xTrain2FScaled, yTrain)
                yPred = svmModel.predict(xTest2FScaled)
                acc = accuracy_score(yTest, yPred)
                print(f"  Kernel={kernel}, C={c}, Gamma={gamma}: Accuracy = {acc:.4f}")
        else:
            svmModel = SVC(kernel=kernel, C=c)
            svmModel.fit(xTrain2FScaled, yTrain)
            yPred = svmModel.predict(xTest2FScaled)
            acc = accuracy_score(yTest, yPred)
            print(f"  Kernel={kernel}, C={c}: Accuracy = {acc:.4f}")

In [ ]:
def plotDecisionBoundary(xData, yData, model, title, featureNames):
    h = 0.05
    xMin, xMax = xData[:, 0].min() - 1, xData[:, 0].max() + 1
    yMin, yMax = xData[:, 1].min() - 1, xData[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(xMin, xMax, h), np.arange(yMin, yMax, h))
    
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    plt.figure(figsize=(10, 6))
    plt.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.viridis)
    
    colors = plt.cm.viridis(np.linspace(0, 1, len(np.unique(yData))))
    for cls, color in zip(np.unique(yData), colors):
        mask = yData == cls
        plt.scatter(xData[mask, 0], xData[mask, 1], c=[color], label=f'Class {cls}', edgecolors='k')
    
    plt.xlabel(featureNames[0])
    plt.ylabel(featureNames[1])
    plt.title(title)
    plt.legend()
    plt.show()

In [ ]:
svmLinear = SVC(kernel='linear', C=1)
svmLinear.fit(xTrain2FScaled, yTrain)
plotDecisionBoundary(xTrain2FScaled, yTrain.values, svmLinear, 'SVM Linear Kernel Decision Boundary', twoFeatures)

In [ ]:
svmRbf = SVC(kernel='rbf', C=1, gamma='scale')
svmRbf.fit(xTrain2FScaled, yTrain)
plotDecisionBoundary(xTrain2FScaled, yTrain.values, svmRbf, 'SVM RBF Kernel Decision Boundary', twoFeatures)

## Task 5: Model Evaluation and Comparison

In [ ]:
knnFinal = KNeighborsClassifier(n_neighbors=5)
knnFinal.fit(xTrainScaled, yTrain)
yPredKnn = knnFinal.predict(xTestScaled)

svmFinal = SVC(kernel='rbf', C=1, gamma='scale')
svmFinal.fit(xTrainScaled, yTrain)
yPredSvm = svmFinal.predict(xTestScaled)

In [ ]:
def evaluateModel(yTrue, yPred, modelName):
    print(f"\n{'='*50}")
    print(f"{modelName} Evaluation:")
    print(f"{'='*50}")
    print(f"Accuracy:  {accuracy_score(yTrue, yPred):.4f}")
    print(f"Precision: {precision_score(yTrue, yPred, average='weighted'):.4f}")
    print(f"Recall:    {recall_score(yTrue, yPred, average='weighted'):.4f}")
    print(f"F1-Score:  {f1_score(yTrue, yPred, average='weighted'):.4f}")
    print(f"\nClassification Report:")
    print(classification_report(yTrue, yPred))
    return confusion_matrix(yTrue, yPred)

cmKnn = evaluateModel(yTest, yPredKnn, "KNN (K=5)")
cmSvm = evaluateModel(yTest, yPredSvm, "SVM (RBF Kernel)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im1 = axes[0].imshow(cmKnn, interpolation='nearest', cmap=plt.cm.Blues)
axes[0].set_title('KNN Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
plt.colorbar(im1, ax=axes[0])

im2 = axes[1].imshow(cmSvm, interpolation='nearest', cmap=plt.cm.Blues)
axes[1].set_title('SVM Confusion Matrix')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
plt.colorbar(im2, ax=axes[1])

plt.tight_layout()
plt.show()

In [ ]:
print("\n" + "="*60)
print("Model Comparison Summary")
print("="*60)
knnAcc = accuracy_score(yTest, yPredKnn)
svmAcc = accuracy_score(yTest, yPredSvm)
print(f"KNN Accuracy: {knnAcc:.4f}")
print(f"SVM Accuracy: {svmAcc:.4f}")
print(f"\nBetter Model: {'SVM' if svmAcc > knnAcc else 'KNN'}")
print("\nAnalysis:")
print("- SVM with RBF kernel handles non-linear decision boundaries well")
print("- KNN is simple and intuitive but sensitive to K value selection")
print("- Both models benefit from feature scaling (standardization)")
print("- SVM typically performs better with high-dimensional data")

## Task 6: Saving Predictions for Analysis

In [ ]:
knnResults = pd.DataFrame({
    'Actual': yTest.values,
    'Predicted': yPredKnn
})
knnResults.to_excel('knn_predictions.xlsx', index=False)
print("KNN predictions saved to knn_predictions.xlsx")

svmResults = pd.DataFrame({
    'Actual': yTest.values,
    'Predicted': yPredSvm
})
svmResults.to_excel('svm_predictions.xlsx', index=False)
print("SVM predictions saved to svm_predictions.xlsx")

In [ ]:
print("\nKNN Predictions Preview:")
print(knnResults.head(10))
print("\nSVM Predictions Preview:")
print(svmResults.head(10))